# MAMMQA with Qwen instead of OpenAI GPT

This notebook rebuilds the full MAMMQA experiment pipeline using Qwen through DashScope:

1. **Stage 1: Modality specialists** for text, tables, and images.
2. **Stage 2: Cross-modal refinement agents** for text-anchored, table-anchored, and image-anchored verification.
3. **Stage 3: Aggregator agent** that produces the final answer.
4. **Baselines**: zero-shot and chain-of-thought.
5. **Tree-of-Thoughts**: candidate thought generation, scoring, DFS pruning.
6. **Evaluation**: exact match, token F1, and lemma-match accuracy.

The code uses the `openai` Python SDK only because DashScope exposes an OpenAI-compatible endpoint. The API key is `DASHSCOPE_API_KEY`, and the base URL points to DashScope/Qwen. No OpenAI API key is used.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -U openai python-dotenv pandas tabulate requests pillow spacy

## 2. Write the pipeline modules

These cells make the notebook self-contained. Running them creates the Python files used by the rest of the notebook.

In [ ]:
%%writefile qwen_client.py
"""Qwen/DashScope client utilities for the MAMMQA pipeline.

This file uses the OpenAI Python SDK only as an OpenAI-compatible HTTP client.
Requests go to DashScope/Qwen because the client is created with a DashScope
base_url and a DASHSCOPE_API_KEY. It does not use an OpenAI API key.
"""

from __future__ import annotations

import base64
import mimetypes
import os
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

from openai import OpenAI
try:
    from openai import APIConnectionError, APIStatusError, AuthenticationError, BadRequestError, RateLimitError
except Exception:  # pragma: no cover - older SDK fallback
    APIConnectionError = APIStatusError = AuthenticationError = BadRequestError = RateLimitError = Exception


QWEN_REGION_BASE_URLS: Dict[str, str] = {
    "beijing": "https://dashscope.aliyuncs.com/compatible-mode/v1",
    "china": "https://dashscope.aliyuncs.com/compatible-mode/v1",
    "cn": "https://dashscope.aliyuncs.com/compatible-mode/v1",
    "us": "https://dashscope-us.aliyuncs.com/compatible-mode/v1",
    "virginia": "https://dashscope-us.aliyuncs.com/compatible-mode/v1",
    "intl": "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    "international": "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    "singapore": "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
}

DEFAULT_TEXT_MODEL = os.getenv("QWEN_TEXT_MODEL", "qwen-plus")
DEFAULT_VISION_MODEL = os.getenv("QWEN_VISION_MODEL", "qwen3-vl-plus")
DEFAULT_REGION = os.getenv("QWEN_REGION", "intl")
DEFAULT_BASE_URL = os.getenv("QWEN_BASE_URL") or QWEN_REGION_BASE_URLS.get(DEFAULT_REGION.lower(), QWEN_REGION_BASE_URLS["intl"])


def get_dashscope_api_key(secret_name: str = "DASHSCOPE_API_KEY") -> Optional[str]:
    """Return DASHSCOPE_API_KEY from env or Kaggle Secrets when available."""
    key = os.getenv(secret_name)
    if key:
        return key
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore

        return UserSecretsClient().get_secret(secret_name)
    except Exception:
        return None


def make_qwen_client(
    api_key: Optional[str] = None,
    base_url: Optional[str] = None,
    region: Optional[str] = None,
) -> OpenAI:
    """Create an OpenAI-compatible client pointed at DashScope/Qwen."""
    key = api_key or get_dashscope_api_key()
    if not key or key in {"sk-YOUR-KEY-HERE", "YOUR_DASHSCOPE_API_KEY"}:
        raise RuntimeError(
            "DASHSCOPE_API_KEY is missing. Add it as an environment variable or Kaggle Secret named DASHSCOPE_API_KEY."
        )
    if base_url is None:
        if region:
            base_url = QWEN_REGION_BASE_URLS.get(region.lower())
            if base_url is None:
                raise ValueError(f"Unknown Qwen region {region!r}. Choose one of {sorted(QWEN_REGION_BASE_URLS)} or pass base_url.")
        else:
            base_url = DEFAULT_BASE_URL
    return OpenAI(api_key=key, base_url=base_url)


def is_data_url(value: str) -> bool:
    return isinstance(value, str) and value.startswith("data:")


def is_http_url(value: str) -> bool:
    return isinstance(value, str) and (value.startswith("http://") or value.startswith("https://"))


def encode_image_file_to_data_url(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image file not found: {path}")
    mime, _ = mimetypes.guess_type(str(path))
    if not mime:
        suffix = path.suffix.lower().lstrip(".") or "jpeg"
        mime = f"image/{'jpeg' if suffix in {'jpg', 'jpeg'} else suffix}"
    data = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{data}"


def image_to_message_part(image: str | Path | Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Convert a URL/data-url/local-path/dict to OpenAI-compatible image_url part."""
    if not image:
        return None
    if isinstance(image, dict):
        if image.get("type") == "image_url":
            return image
        url = image.get("url") or image.get("image_url") or image.get("path")
        if isinstance(url, dict):
            url = url.get("url")
        if not url:
            return None
        image = url
    s = str(image)
    # Ignore synthetic placeholders like "[Image: title]".
    if s.startswith("[Image:"):
        return None
    if is_http_url(s) or is_data_url(s):
        return {"type": "image_url", "image_url": {"url": s}}
    p = Path(s)
    if p.exists() and p.is_file():
        return {"type": "image_url", "image_url": {"url": encode_image_file_to_data_url(p)}}
    return None


def images_to_message_parts(images: Optional[Sequence[str | Path | Dict[str, Any]]]) -> List[Dict[str, Any]]:
    if not images:
        return []
    parts: List[Dict[str, Any]] = []
    for image in images:
        part = image_to_message_part(image)
        if part is not None:
            parts.append(part)
    return parts


def make_content(text: str, images: Optional[Sequence[str | Path | Dict[str, Any]]] = None) -> str | List[Dict[str, Any]]:
    """Return string content for text-only calls, or multimodal content if images exist."""
    img_parts = images_to_message_parts(images)
    if not img_parts:
        return text
    return [{"type": "text", "text": text}] + img_parts


def _error_text(exc: BaseException) -> str:
    pieces = [str(exc)]
    response = getattr(exc, "response", None)
    if response is not None:
        try:
            pieces.append(response.text)
        except Exception:
            pass
    body = getattr(exc, "body", None)
    if body:
        pieces.append(str(body))
    return "\n".join(pieces)


def _is_fatal_error(exc: BaseException) -> bool:
    msg = _error_text(exc).lower()
    fatal_markers = [
        "invalid api key",
        "invalid_api_key",
        "authentication",
        "unauthorized",
        "permission",
        "insufficient_quota",
        "quota",
        "billing",
        "access denied",
        "model not found",
        "does not exist",
        "not supported",
    ]
    return any(marker in msg for marker in fatal_markers) and "rate" not in msg


def extract_response_text(response: Any) -> str:
    """Extract assistant text from a Chat Completions response or stream aggregate."""
    try:
        content = response.choices[0].message.content
        if isinstance(content, list):
            return "".join(str(x.get("text", "")) if isinstance(x, dict) else str(x) for x in content)
        return content or ""
    except Exception:
        return str(response)


def call_qwen(
    client: OpenAI,
    messages: List[Dict[str, Any]],
    model: str = DEFAULT_TEXT_MODEL,
    temperature: float = 0.2,
    top_p: float = 0.8,
    max_tokens: int = 700,
    max_retries: int = 5,
    timeout: Optional[float] = None,
    verbose: bool = False,
    **extra: Any,
) -> str:
    """Call Qwen through DashScope OpenAI-compatible chat completions."""
    for attempt in range(max_retries + 1):
        try:
            kwargs: Dict[str, Any] = dict(
                model=model,
                messages=messages,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
            )
            if timeout is not None:
                kwargs["timeout"] = timeout
            kwargs.update(extra)
            response = client.chat.completions.create(**kwargs)
            return extract_response_text(response)
        except Exception as exc:  # noqa: BLE001 - preserve rich API exceptions from compatible services
            msg = _error_text(exc)
            if _is_fatal_error(exc):
                raise RuntimeError(
                    "Qwen/DashScope returned a non-retryable error. Check DASHSCOPE_API_KEY, QWEN_BASE_URL/region, "
                    f"model name, quota, and account permissions. Original error:\n{msg}"
                ) from exc
            if attempt >= max_retries:
                raise RuntimeError(f"Qwen call failed after {max_retries} retries. Last error:\n{msg}") from exc
            wait = min(90.0, (2 ** attempt) + random.uniform(0.0, 2.0))
            if verbose:
                print(f"Qwen temporary error; retry {attempt + 1}/{max_retries} after {wait:.1f}s")
            time.sleep(wait)
    raise AssertionError("unreachable")


In [ ]:
%%writefile Dataloader.py
"""MultimodalQA dataloader used by the Qwen MAMMQA pipeline."""

from __future__ import annotations

import base64
import json
import mimetypes
import os
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

import pandas as pd


def _unwrap_kaggle_file(path: str | Path) -> Optional[str]:
    """Kaggle sometimes mounts a file as a directory containing the real file."""
    p = Path(path)
    if p.is_file():
        return str(p)
    if p.is_dir():
        files = [x for x in p.iterdir() if x.is_file()]
        if len(files) == 1:
            return str(files[0])
        for ext in (".jsonl", ".json"):
            hits = [x for x in files if x.name.endswith(ext)]
            if hits:
                return str(hits[0])
    return None


def find_file(root: str | Path, names: Sequence[str]) -> Optional[str]:
    root = Path(root)
    if not root.exists():
        return None
    for name in names:
        direct = _unwrap_kaggle_file(root / name)
        if direct:
            return direct
    wanted = set(names)
    for p in root.rglob("*"):
        if p.name in wanted:
            got = _unwrap_kaggle_file(p)
            if got:
                return got
    return None


def discover_mmqa_files(root: str | Path) -> Dict[str, Optional[str]]:
    return {
        "dev": find_file(root, ["MMQA_dev.jsonl", "dev.jsonl", "dev.json", "MMQA_dev.json"]),
        "texts": find_file(root, ["MMQA_texts.jsonl", "texts.jsonl", "texts.json", "MMQA_texts.json"]),
        "tables": find_file(root, ["MMQA_tables.jsonl", "tables.jsonl", "tables.json", "MMQA_tables.json"]),
        "images": find_file(root, ["MMQA_images.jsonl", "images.jsonl", "images.json", "MMQA_images.json"]),
    }


def load_records(path: str | Path) -> List[Dict[str, Any]]:
    path = Path(path)
    records: List[Dict[str, Any]] = []
    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]
    if isinstance(data, dict):
        for key in ("data", "examples", "records", "items"):
            if isinstance(data.get(key), list):
                return [x for x in data[key] if isinstance(x, dict)]
        # Fall back to dict values when it looks like an id -> record mapping.
        if all(isinstance(v, dict) for v in data.values()):
            return list(data.values())
    raise ValueError(f"Unsupported JSON structure in {path}")


def record_id(record: Dict[str, Any]) -> Optional[str]:
    for key in ("id", "uid", "doc_id", "table_id", "text_doc_id", "image_doc_id", "filename", "file_name"):
        value = record.get(key)
        if value is not None:
            return str(value)
    return None


def make_index(records: Iterable[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    for r in records:
        rid = record_id(r)
        if rid:
            out[rid] = r
    return out


def make_columns_unique(df: pd.DataFrame) -> pd.DataFrame:
    counts: Dict[str, int] = {}
    new_cols: List[str] = []
    for col in [str(c) for c in df.columns]:
        if col in counts:
            counts[col] += 1
            new_cols.append(f"{col}_{counts[col]}")
        else:
            counts[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df


def _cell_text(cell: Any) -> str:
    if isinstance(cell, dict):
        return str(cell.get("text") or cell.get("value") or cell.get("content") or "")
    return str(cell)


def parse_table_record(record: Optional[Dict[str, Any]]) -> str:
    if not record:
        return "No table data"
    title = record.get("title") or record.get("table_title") or "Untitled table"
    table_obj = record.get("table", record)

    df: Optional[pd.DataFrame] = None
    try:
        if isinstance(table_obj, dict) and "header" in table_obj and "table_rows" in table_obj:
            headers = [_cell_text(h.get("column_name", h) if isinstance(h, dict) else h) for h in table_obj.get("header", [])]
            rows = [[_cell_text(cell) for cell in row] for row in table_obj.get("table_rows", [])]
            if rows:
                if headers and len(headers) == len(rows[0]):
                    df = pd.DataFrame(rows, columns=headers)
                else:
                    df = pd.DataFrame(rows)
        elif isinstance(table_obj, dict) and "rows" in table_obj:
            headers = table_obj.get("headers") or table_obj.get("header") or []
            rows = table_obj.get("rows") or []
            rows = [[_cell_text(cell) for cell in row] for row in rows]
            if rows:
                df = pd.DataFrame(rows, columns=headers if headers and len(headers) == len(rows[0]) else None)
        elif isinstance(table_obj, list):
            df = pd.DataFrame(table_obj)
    except Exception:
        df = None

    if df is None or df.empty:
        raw = record.get("text") or record.get("markdown") or record.get("html") or str(record)[:4000]
        return f"Table title: {title}\n{raw}"
    df = make_columns_unique(df)
    try:
        md = df.to_markdown(index=False)
    except Exception:
        md = df.to_csv(index=False)
    return f"Table title: {title}\n{md}"


def format_text_record(record: Dict[str, Any]) -> str:
    title = record.get("title") or record.get("page_title") or record.get("name") or "Untitled text"
    text = record.get("text") or record.get("contents") or record.get("passage") or record.get("content") or ""
    return f"title: {title}\ntext: {text}"


def _as_list(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    if isinstance(value, tuple):
        return [str(x) for x in value]
    return [str(value)]


def _answer_to_str(answer: Any) -> str:
    if answer is None:
        return ""
    if isinstance(answer, list):
        return " | ".join(str(x) for x in answer)
    if isinstance(answer, dict):
        for key in ("answer", "text", "value"):
            if key in answer:
                return _answer_to_str(answer[key])
    return str(answer)


def _file_to_data_url(path: str | Path) -> str:
    path = Path(path)
    mime, _ = mimetypes.guess_type(str(path))
    if not mime:
        mime = "image/jpeg"
    return f"data:{mime};base64," + base64.b64encode(path.read_bytes()).decode("utf-8")


class MultiModalQADataLoader:
    """Load MultimodalQA dev/text/table/image files and create agent inputs."""

    def __init__(
        self,
        dev_file: str,
        tables_file: str,
        texts_file: str,
        images_file: Optional[str] = None,
        images_base_url: Optional[str] = None,
        encode_images: bool = False,
    ) -> None:
        self.dev_file = _unwrap_kaggle_file(dev_file) or dev_file
        self.tables_file = _unwrap_kaggle_file(tables_file) or tables_file
        self.texts_file = _unwrap_kaggle_file(texts_file) or texts_file
        self.images_file = (_unwrap_kaggle_file(images_file) if images_file else None) or images_file
        self.images_base_url = images_base_url
        self.encode_images = encode_images

        self.dev = load_records(self.dev_file)
        self.tables = load_records(self.tables_file)
        self.texts = load_records(self.texts_file)
        self.images = load_records(self.images_file) if self.images_file else []

        self.table_lookup = make_index(self.tables)
        self.text_lookup = make_index(self.texts)
        self.image_lookup = make_index(self.images)

    @classmethod
    def from_root(cls, root: str | Path, images_base_url: Optional[str] = None, encode_images: bool = False) -> "MultiModalQADataLoader":
        root = Path(root)
        files = discover_mmqa_files(root)
        missing = [k for k in ("dev", "texts", "tables") if not files.get(k)]
        if missing:
            raise FileNotFoundError(f"Could not find required MMQA files {missing} under {root}. Found: {files}")
        if images_base_url is None:
            for candidate in (root / "images", root / "MMQA_images", root):
                if candidate.exists():
                    images_base_url = str(candidate)
                    break
        return cls(
            dev_file=files["dev"],
            tables_file=files["tables"],
            texts_file=files["texts"],
            images_file=files.get("images"),
            images_base_url=images_base_url,
            encode_images=encode_images,
        )

    def __len__(self) -> int:
        return len(self.dev)

    def _resolve_image_record(self, image_id: str) -> Optional[str]:
        rec = self.image_lookup.get(str(image_id))
        if not rec:
            return None
        for key in ("url", "image_url", "src"):
            if rec.get(key):
                return str(rec[key])
        path_value = rec.get("path") or rec.get("file_name") or rec.get("filename") or rec.get("image")
        if path_value:
            p = Path(str(path_value))
            if not p.is_absolute() and self.images_base_url:
                p = Path(self.images_base_url) / p
            if p.exists():
                return _file_to_data_url(p) if self.encode_images else str(p)
            s = str(path_value)
            if s.startswith("http://") or s.startswith("https://") or s.startswith("data:"):
                return s
        title = rec.get("title") or rec.get("page_title")
        if title:
            return f"[Image: {title}]"
        return None

    def get_agent_inputs(self, index: int) -> Dict[str, Any]:
        entry = self.dev[index]
        meta = entry.get("metadata") or entry.get("meta") or {}
        question = entry.get("question") or entry.get("query") or ""
        qid = entry.get("id") or entry.get("qid") or entry.get("question_id") or str(index)
        qtype = entry.get("type") or entry.get("question_type") or meta.get("type") or meta.get("question_type") or "unknown"

        answer = entry.get("answer")
        if answer is None:
            answer = entry.get("answers")
        if answer is None:
            answer = meta.get("answer") or meta.get("answers")

        text_ids = _as_list(meta.get("text_doc_ids") or meta.get("text_ids") or entry.get("text_doc_ids") or entry.get("text_ids"))
        table_ids = _as_list(meta.get("table_id") or meta.get("table_doc_ids") or meta.get("table_ids") or entry.get("table_id"))
        image_ids = _as_list(meta.get("image_doc_ids") or meta.get("image_ids") or entry.get("image_doc_ids") or entry.get("image_ids"))

        text_chunks = [format_text_record(self.text_lookup[tid]) for tid in text_ids if tid in self.text_lookup]
        text = "\n\n".join(text_chunks) if text_chunks else "No text available"

        table_chunks = [parse_table_record(self.table_lookup[tid]) for tid in table_ids if tid in self.table_lookup]
        table = "\n\n".join(table_chunks) if table_chunks else "No table data"

        images: List[str] = []
        for iid in image_ids:
            resolved = self._resolve_image_record(iid)
            if resolved:
                images.append(resolved)

        modalities = {
            "text": text != "No text available",
            "table": table != "No table data",
            "image": bool([x for x in images if not str(x).startswith("[Image:")]),
        }
        return {
            "id": str(qid),
            "question": question,
            "answer": _answer_to_str(answer),
            "raw_answer": answer,
            "type": qtype,
            "text": text,
            "table": table,
            "images": images,
            "metadata": meta,
            "modalities": modalities,
        }


In [ ]:
%%writefile agents_qwen.py
"""Qwen implementation of MAMMQA agents, baselines, and experiment runner."""

from __future__ import annotations

import json
import re
from typing import Any, Dict, Iterable, List, Optional, Sequence

from qwen_client import DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL, call_qwen, images_to_message_parts, make_content


TEMPERATURE = 0.2
TOP_P = 0.8
MAX_TOKENS = 700


def _clip(value: Any, max_chars: int = 12000) -> str:
    s = "" if value is None else str(value)
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + f"\n...[truncated {len(s) - max_chars} chars]"


def extract_answer_tag(text: str) -> str:
    if not text:
        return ""
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", str(text), flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()
    # Also support JSON-like outputs.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "answer" in obj:
            return str(obj["answer"]).strip()
    except Exception:
        pass
    return str(text).strip()


AGENT_STAGE1_SYSTEM_PROMPT = """
You are a modality-specialist agent for multimodal question answering.
You receive ONE modality plus a question. Do not guess beyond the provided input.

Tasks:
1. Identify the modality: text, table, or image.
2. Extract only evidence relevant to the question.
3. Break the question into smaller sub-questions if useful.
4. State whether this modality is sufficient, insufficient, or contradictory.
5. Give a short candidate answer only if directly supported.

Output format:
<modality>...</modality>
<evidence>bullet list of grounded evidence</evidence>
<subquestions>bullet list if needed</subquestions>
<candidate_answer>answer or NOT_ENOUGH_INFORMATION</candidate_answer>
<confidence>low/medium/high</confidence>
""".strip()


AGENT_STAGE2_SYSTEM_PROMPT = """
You are a cross-modal refinement agent.
You receive one modality-specialist insight as the anchor, plus other available raw modalities.

Tasks:
1. Verify the anchor insight against the other modalities.
2. Add missing evidence from the other modalities.
3. Resolve conflicts and say which evidence is strongest.
4. Produce a refined candidate answer.

Use only the supplied evidence. If evidence is missing, say so.

Output format:
<verified_evidence>...</verified_evidence>
<conflicts>...</conflicts>
<refined_reasoning>...</refined_reasoning>
<candidate_answer>answer or NOT_ENOUGH_INFORMATION</candidate_answer>
<confidence>low/medium/high</confidence>
""".strip()


AGGREGATOR_SYSTEM_PROMPT = """
You are the final MAMMQA aggregator agent.
You receive outputs from modality specialists and cross-modal refinement agents.

Tasks:
1. Compare all candidate answers and confidence levels.
2. Prefer answers grounded in explicit evidence.
3. Resolve contradictions conservatively.
4. Return a concise final answer.

Output format exactly:
<reasoning>brief evidence-based reasoning</reasoning>
<answer>final answer only</answer>
""".strip()


ZS_SYSTEM_PROMPT = """
Answer the question directly. If you are unsure, give the best concise answer.
Output format exactly:
<answer>final answer only</answer>
""".strip()


COT_SYSTEM_PROMPT = """
You are given a question and specific multimodal evidence. Answer using ONLY the provided evidence.
Think step by step, then give the answer.
Output format exactly:
<reasoning>brief reasoning grounded in the evidence</reasoning>
<answer>final answer only</answer>
""".strip()


def _call_text(client: Any, system: str, user: str, model: Optional[str] = None, **kwargs: Any) -> str:
    return call_qwen(
        client,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        model=model or DEFAULT_TEXT_MODEL,
        temperature=kwargs.pop("temperature", TEMPERATURE),
        top_p=kwargs.pop("top_p", TOP_P),
        max_tokens=kwargs.pop("max_tokens", MAX_TOKENS),
        **kwargs,
    )


def _call_maybe_vision(
    client: Any,
    system: str,
    user_text: str,
    images: Optional[Sequence[Any]] = None,
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    **kwargs: Any,
) -> str:
    image_parts = images_to_message_parts(images)
    content = make_content(user_text, images) if image_parts else user_text
    model = (vision_model or DEFAULT_VISION_MODEL) if image_parts else (text_model or DEFAULT_TEXT_MODEL)
    return call_qwen(
        client,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": content}],
        model=model,
        temperature=kwargs.pop("temperature", TEMPERATURE),
        top_p=kwargs.pop("top_p", TOP_P),
        max_tokens=kwargs.pop("max_tokens", MAX_TOKENS),
        **kwargs,
    )


# ----------------------------- Stage 1 agents -----------------------------

def text_agent(client: Any, question: str, texts: str, model: Optional[str] = None, **kwargs: Any) -> str:
    if not texts or str(texts).strip().lower() == "no text available":
        return "NO_TEXT_AVAILABLE: No text modality was provided."
    user = f"Question:\n{question}\n\nText modality:\n{_clip(texts)}"
    return _call_text(client, AGENT_STAGE1_SYSTEM_PROMPT, user, model=model, **kwargs)


def table_agent(client: Any, question: str, tables: str, model: Optional[str] = None, **kwargs: Any) -> str:
    if not tables or str(tables).strip().lower() == "no table data":
        return "NO_TABLE_AVAILABLE: No table modality was provided."
    user = f"Question:\n{question}\n\nTable modality as markdown:\n{_clip(tables)}"
    return _call_text(client, AGENT_STAGE1_SYSTEM_PROMPT, user, model=model, **kwargs)


def image_agent(client: Any, question: str, images: Optional[Sequence[Any]], model: Optional[str] = None, **kwargs: Any) -> str:
    if not images_to_message_parts(images):
        return "NO_IMAGE_AVAILABLE: No usable image modality was provided."
    user = f"Question:\n{question}\n\nImage modality: the attached image(s). Extract visual evidence relevant to the question."
    return _call_maybe_vision(client, AGENT_STAGE1_SYSTEM_PROMPT, user, images=images, vision_model=model, **kwargs)


# ----------------------------- Stage 2 agents -----------------------------

def text_cross_agent(
    client: Any,
    question: str,
    text_insight: str,
    raw_tables: str,
    raw_images: Optional[Sequence[Any]],
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    **kwargs: Any,
) -> str:
    if text_insight.startswith("NO_TEXT_AVAILABLE"):
        return "TEXT_CROSS_SKIPPED: No text insight exists to refine."
    user = f"""
Question:
{question}

Anchor insight from text specialist:
{_clip(text_insight, 8000)}

Other modality - table:
{_clip(raw_tables)}

Other modality - image(s): attached if available.
""".strip()
    return _call_maybe_vision(client, AGENT_STAGE2_SYSTEM_PROMPT, user, images=raw_images, text_model=text_model, vision_model=vision_model, **kwargs)


def table_cross_agent(
    client: Any,
    question: str,
    table_insight: str,
    raw_text: str,
    raw_images: Optional[Sequence[Any]],
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    **kwargs: Any,
) -> str:
    if table_insight.startswith("NO_TABLE_AVAILABLE"):
        return "TABLE_CROSS_SKIPPED: No table insight exists to refine."
    user = f"""
Question:
{question}

Anchor insight from table specialist:
{_clip(table_insight, 8000)}

Other modality - text:
{_clip(raw_text)}

Other modality - image(s): attached if available.
""".strip()
    return _call_maybe_vision(client, AGENT_STAGE2_SYSTEM_PROMPT, user, images=raw_images, text_model=text_model, vision_model=vision_model, **kwargs)


def image_cross_agent(
    client: Any,
    question: str,
    image_insight: str,
    raw_text: str,
    raw_tables: str,
    raw_images: Optional[Sequence[Any]],
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    **kwargs: Any,
) -> str:
    if image_insight.startswith("NO_IMAGE_AVAILABLE"):
        return "IMAGE_CROSS_SKIPPED: No image insight exists to refine."
    user = f"""
Question:
{question}

Anchor insight from image specialist:
{_clip(image_insight, 8000)}

Other modality - text:
{_clip(raw_text)}

Other modality - table:
{_clip(raw_tables)}

Original image(s): attached again if available.
""".strip()
    return _call_maybe_vision(client, AGENT_STAGE2_SYSTEM_PROMPT, user, images=raw_images, text_model=text_model, vision_model=vision_model, **kwargs)


# ----------------------------- Stage 3 aggregator -----------------------------

def reasoning_agent(
    client: Any,
    question: str,
    text_insight: str,
    table_insight: str,
    image_insight: str,
    text_cross: str,
    table_cross: str,
    image_cross: str,
    model: Optional[str] = None,
    **kwargs: Any,
) -> str:
    user = f"""
Question:
{question}

STAGE 1 - Text specialist:
{_clip(text_insight, 6000)}

STAGE 1 - Table specialist:
{_clip(table_insight, 6000)}

STAGE 1 - Image specialist:
{_clip(image_insight, 6000)}

STAGE 2 - Text-anchored refinement:
{_clip(text_cross, 6000)}

STAGE 2 - Table-anchored refinement:
{_clip(table_cross, 6000)}

STAGE 2 - Image-anchored refinement:
{_clip(image_cross, 6000)}
""".strip()
    return _call_text(client, AGGREGATOR_SYSTEM_PROMPT, user, model=model, max_tokens=kwargs.pop("max_tokens", 500), **kwargs)


def _available_modalities(text: str, tables: str, images: Optional[Sequence[Any]]) -> Dict[str, bool]:
    return {
        "text": bool(text and str(text).strip().lower() != "no text available"),
        "table": bool(tables and str(tables).strip().lower() != "no table data"),
        "image": bool(images_to_message_parts(images)),
    }


def _needed_by_qtype(qtype: Optional[str]) -> Optional[set[str]]:
    if not qtype:
        return None
    q = str(qtype).lower()
    if "compose" in q or "multi" in q or "+" in q:
        return {"text", "table", "image"}
    if "table" in q:
        return {"table"}
    if "image" in q:
        return {"image"}
    if "text" in q:
        return {"text"}
    return None


def get_answer_MM(
    client: Any,
    question: str,
    text: str,
    tables: str,
    images: Optional[Sequence[Any]],
    model: Optional[str] = None,
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    qtype: Optional[str] = None,
    strict_paper_mode: bool = True,
    reduce_modalities_by_question_type: bool = False,
    verbose: bool = False,
    **kwargs: Any,
) -> Dict[str, str]:
    """Run the complete three-stage MAMMQA pipeline.

    strict_paper_mode=True keeps Stage 2 and Stage 3 even for single-modality
    questions. reduce_modalities_by_question_type=True is a cost-saving mode.
    """
    text_model = text_model or model or DEFAULT_TEXT_MODEL
    vision_model = vision_model or DEFAULT_VISION_MODEL
    avail = _available_modalities(text, tables, images)
    needed = set(avail.keys())
    if reduce_modalities_by_question_type:
        by_type = _needed_by_qtype(qtype)
        if by_type:
            needed = by_type
    # Never attempt unavailable modalities.
    needed = {m for m in needed if avail.get(m)}
    if strict_paper_mode and not needed:
        needed = {m for m, ok in avail.items() if ok}

    if verbose:
        print(f"Available modalities: {avail}; running specialists for: {sorted(needed)}")
        print("Stage 1: modality specialists")

    text_insight = text_agent(client, question, text, model=text_model, **kwargs) if "text" in needed else "NO_TEXT_AVAILABLE: Text specialist not run."
    if verbose and "text" in needed: print("  done text specialist")
    table_insight = table_agent(client, question, tables, model=text_model, **kwargs) if "table" in needed else "NO_TABLE_AVAILABLE: Table specialist not run."
    if verbose and "table" in needed: print("  done table specialist")
    image_insight = image_agent(client, question, images, model=vision_model, **kwargs) if "image" in needed else "NO_IMAGE_AVAILABLE: Image specialist not run."
    if verbose and "image" in needed: print("  done image specialist")

    if verbose:
        print("Stage 2: cross-modal refinement")
    # Full paper mode still performs refinement for every modality that has a Stage 1 insight.
    text_cross = text_cross_agent(client, question, text_insight, tables, images, text_model=text_model, vision_model=vision_model, **kwargs) if "text" in needed else "TEXT_CROSS_SKIPPED: Text specialist not run."
    if verbose and "text" in needed: print("  done text-anchored refinement")
    table_cross = table_cross_agent(client, question, table_insight, text, images, text_model=text_model, vision_model=vision_model, **kwargs) if "table" in needed else "TABLE_CROSS_SKIPPED: Table specialist not run."
    if verbose and "table" in needed: print("  done table-anchored refinement")
    image_cross = image_cross_agent(client, question, image_insight, text, tables, images, text_model=text_model, vision_model=vision_model, **kwargs) if "image" in needed else "IMAGE_CROSS_SKIPPED: Image specialist not run."
    if verbose and "image" in needed: print("  done image-anchored refinement")

    if verbose:
        print("Stage 3: aggregator")
    final = reasoning_agent(
        client,
        question,
        text_insight,
        table_insight,
        image_insight,
        text_cross,
        table_cross,
        image_cross,
        model=text_model,
        **kwargs,
    )

    return {
        "Text Agent Output": text_insight,
        "Table Agent Output": table_insight,
        "Image Agent Output": image_insight,
        "Text Cross Agent Output": text_cross,
        "Table Cross Agent Output": table_cross,
        "Image Cross Agent Output": image_cross,
        "Final Answer": final,
        "predicted_answer": extract_answer_tag(final),
        "available_modalities": json.dumps(avail),
        "ran_modalities": json.dumps(sorted(needed)),
    }


# ----------------------------- Baselines -----------------------------

def get_answer_zs_no_data(client: Any, question: str, model: Optional[str] = None, **kwargs: Any) -> str:
    return _call_text(client, ZS_SYSTEM_PROMPT, f"Question:\n{question}", model=model or DEFAULT_TEXT_MODEL, max_tokens=kwargs.pop("max_tokens", 250), **kwargs)


def get_answer_cot(
    client: Any,
    question: str,
    text: str,
    table: str,
    images: Optional[Sequence[Any]],
    model: Optional[str] = None,
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    **kwargs: Any,
) -> str:
    user = f"""
Question:
{question}

Text evidence:
{_clip(text)}

Table evidence:
{_clip(table)}

Image evidence: attached if available.
""".strip()
    return _call_maybe_vision(
        client,
        COT_SYSTEM_PROMPT,
        user,
        images=images,
        text_model=text_model or model or DEFAULT_TEXT_MODEL,
        vision_model=vision_model or DEFAULT_VISION_MODEL,
        max_tokens=kwargs.pop("max_tokens", 700),
        **kwargs,
    )


def get_answer_Many(
    client: Any,
    examples: Iterable[Dict[str, Any]],
    method: str = "mammqa",
    text_model: Optional[str] = None,
    vision_model: Optional[str] = None,
    verbose: bool = False,
    **kwargs: Any,
) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for i, ex in enumerate(examples):
        question = ex["question"]
        if verbose:
            print(f"\nExample {i + 1}: {question}")
        if method == "mammqa":
            res = get_answer_MM(
                client,
                question=question,
                text=ex.get("text", ""),
                tables=ex.get("table", ""),
                images=ex.get("images", []),
                text_model=text_model,
                vision_model=vision_model,
                qtype=ex.get("type"),
                verbose=verbose,
                **kwargs,
            )
            predicted = res.get("predicted_answer", "")
        elif method == "cot":
            raw = get_answer_cot(client, question, ex.get("text", ""), ex.get("table", ""), ex.get("images", []), text_model=text_model, vision_model=vision_model, **kwargs)
            res = {"Final Answer": raw}
            predicted = extract_answer_tag(raw)
        elif method == "zs":
            raw = get_answer_zs_no_data(client, question, model=text_model, **kwargs)
            res = {"Final Answer": raw}
            predicted = extract_answer_tag(raw)
        else:
            raise ValueError("method must be one of: mammqa, cot, zs")
        record = {
            "id": ex.get("id", str(i)),
            "question": question,
            "type": ex.get("type", ""),
            "method": method,
            "predicted_answer": predicted,
            "gold_answer": ex.get("answer", ""),
            **res,
        }
        records.append(record)
    return records


In [ ]:
%%writefile treeofthoughts_qwen.py
"""Tree-of-Thoughts utilities for Qwen MAMMQA experiments."""

from __future__ import annotations

import json
import re
from typing import Any, Dict, List, Optional, Sequence

from qwen_client import DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL, call_qwen, images_to_message_parts, make_content


def _clip(value: Any, max_chars: int = 10000) -> str:
    s = "" if value is None else str(value)
    return s if len(s) <= max_chars else s[:max_chars] + "\n...[truncated]"


def _parse_json_any(text: str) -> Any:
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    m = re.search(r"(\[.*\]|\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    return None


def _qwen_evidence_call(
    client: Any,
    system: str,
    user_text: str,
    images: Optional[Sequence[Any]],
    text_model: str,
    vision_model: str,
    max_tokens: int = 700,
    temperature: float = 0.4,
) -> str:
    has_images = bool(images_to_message_parts(images))
    content = make_content(user_text, images) if has_images else user_text
    model = vision_model if has_images else text_model
    return call_qwen(
        client,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": content}],
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
    )


def generate_initial_thoughts(
    client: Any,
    question: str,
    text: str = "",
    table: str = "",
    images: Optional[Sequence[Any]] = None,
    k: int = 5,
    text_model: str = DEFAULT_TEXT_MODEL,
    vision_model: str = DEFAULT_VISION_MODEL,
) -> List[Dict[str, Any]]:
    system = """
Generate diverse initial reasoning thoughts for multimodal QA.
Each thought should be a possible evidence path, not just a final answer.
Return strict JSON: a list of objects with keys thought, evidence_needed, candidate_answer.
""".strip()
    user = f"""
Question: {question}

Text:
{_clip(text)}

Table:
{_clip(table)}

Images: attached if available.

Generate exactly {k} thoughts.
""".strip()
    out = _qwen_evidence_call(client, system, user, images, text_model, vision_model, max_tokens=900, temperature=0.7)
    parsed = _parse_json_any(out)
    if isinstance(parsed, list):
        thoughts = [x if isinstance(x, dict) else {"thought": str(x)} for x in parsed]
    else:
        lines = [ln.strip(" -0123456789.") for ln in out.splitlines() if ln.strip()]
        thoughts = [{"thought": ln} for ln in lines[:k]]
    return thoughts[:k]


def score_thought(
    client: Any,
    question: str,
    thought: Dict[str, Any] | str,
    text: str = "",
    table: str = "",
    images: Optional[Sequence[Any]] = None,
    text_model: str = DEFAULT_TEXT_MODEL,
    vision_model: str = DEFAULT_VISION_MODEL,
) -> Dict[str, Any]:
    system = """
Score the usefulness of a thought for answering the question from the provided evidence.
Return strict JSON with keys score (0 to 1) and rationale.
""".strip()
    user = f"""
Question: {question}
Thought: {json.dumps(thought, ensure_ascii=False)}

Text:
{_clip(text, 7000)}

Table:
{_clip(table, 7000)}

Images: attached if available.
""".strip()
    out = _qwen_evidence_call(client, system, user, images, text_model, vision_model, max_tokens=300, temperature=0.1)
    parsed = _parse_json_any(out)
    if isinstance(parsed, dict):
        try:
            parsed["score"] = float(parsed.get("score", 0.0))
        except Exception:
            parsed["score"] = 0.0
        parsed["raw"] = out
        return parsed
    m = re.search(r"(?:score|rating)\D+([01](?:\.\d+)?)", out, flags=re.IGNORECASE)
    return {"score": float(m.group(1)) if m else 0.0, "rationale": out, "raw": out}


def expand_thought(
    client: Any,
    question: str,
    path: List[Dict[str, Any]],
    text: str = "",
    table: str = "",
    images: Optional[Sequence[Any]] = None,
    k: int = 3,
    text_model: str = DEFAULT_TEXT_MODEL,
    vision_model: str = DEFAULT_VISION_MODEL,
) -> List[Dict[str, Any]]:
    system = """
Expand the current reasoning path with the next useful reasoning steps.
Return strict JSON: a list of objects with keys thought, operation, candidate_answer.
""".strip()
    user = f"""
Question: {question}
Current path:
{json.dumps(path, ensure_ascii=False, indent=2)}

Text:
{_clip(text, 7000)}

Table:
{_clip(table, 7000)}

Images: attached if available.

Generate exactly {k} next thoughts.
""".strip()
    out = _qwen_evidence_call(client, system, user, images, text_model, vision_model, max_tokens=800, temperature=0.6)
    parsed = _parse_json_any(out)
    if isinstance(parsed, list):
        return [x if isinstance(x, dict) else {"thought": str(x)} for x in parsed][:k]
    lines = [ln.strip(" -0123456789.") for ln in out.splitlines() if ln.strip()]
    return [{"thought": ln} for ln in lines[:k]]


def final_answer_from_path(
    client: Any,
    question: str,
    path: List[Dict[str, Any]],
    text: str = "",
    table: str = "",
    images: Optional[Sequence[Any]] = None,
    text_model: str = DEFAULT_TEXT_MODEL,
    vision_model: str = DEFAULT_VISION_MODEL,
) -> str:
    system = """
Use the selected reasoning path and evidence to produce the final answer.
Output format exactly:
<reasoning>brief reasoning</reasoning>
<answer>final answer only</answer>
""".strip()
    user = f"""
Question: {question}
Selected reasoning path:
{json.dumps(path, ensure_ascii=False, indent=2)}

Text:
{_clip(text, 8000)}

Table:
{_clip(table, 8000)}

Images: attached if available.
""".strip()
    return _qwen_evidence_call(client, system, user, images, text_model, vision_model, max_tokens=600, temperature=0.2)


In [ ]:
%%writefile tot_dfs_qwen.py
"""DFS driver for Qwen Tree-of-Thoughts experiments."""

from __future__ import annotations

from typing import Any, Dict, List, Optional, Sequence

from agents_qwen import extract_answer_tag
from qwen_client import DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL
from treeofthoughts_qwen import expand_thought, final_answer_from_path, generate_initial_thoughts, score_thought


def run_dfs(
    client: Any,
    question: str,
    text: str = "",
    table: str = "",
    images: Optional[Sequence[Any]] = None,
    initial_thoughts: Optional[List[Dict[str, Any]]] = None,
    k: int = 3,
    beam_threshold: float = 0.45,
    max_depth: int = 3,
    text_model: str = DEFAULT_TEXT_MODEL,
    vision_model: str = DEFAULT_VISION_MODEL,
    verbose: bool = False,
) -> Dict[str, Any]:
    """Run a small DFS Tree-of-Thoughts search with pruning."""
    if initial_thoughts is None:
        initial_thoughts = generate_initial_thoughts(
            client, question, text=text, table=table, images=images, k=k, text_model=text_model, vision_model=vision_model
        )

    best: Dict[str, Any] = {"score": -1.0, "path": [], "final_raw": "", "predicted_answer": ""}

    def consider_final(path: List[Dict[str, Any]], score: float) -> None:
        nonlocal best
        raw = final_answer_from_path(
            client, question, path, text=text, table=table, images=images, text_model=text_model, vision_model=vision_model
        )
        if score > best["score"]:
            best = {"score": score, "path": path, "final_raw": raw, "predicted_answer": extract_answer_tag(raw)}

    def dfs(path: List[Dict[str, Any]], depth: int) -> None:
        if not path:
            return
        scored = score_thought(
            client, question, path[-1], text=text, table=table, images=images, text_model=text_model, vision_model=vision_model
        )
        score = float(scored.get("score", 0.0))
        path[-1]["score"] = score
        path[-1]["score_rationale"] = scored.get("rationale", "")
        if verbose:
            print(f"depth={depth} score={score:.2f} thought={str(path[-1])[:120]}")
        if score < beam_threshold:
            return
        if depth >= max_depth:
            consider_final(path, score)
            return
        children = expand_thought(
            client, question, path, text=text, table=table, images=images, k=k, text_model=text_model, vision_model=vision_model
        )
        # Score children before recursing, keep top k.
        scored_children = []
        for child in children:
            child_score = score_thought(
                client, question, child, text=text, table=table, images=images, text_model=text_model, vision_model=vision_model
            )
            child["score"] = float(child_score.get("score", 0.0))
            child["score_rationale"] = child_score.get("rationale", "")
            scored_children.append(child)
        scored_children.sort(key=lambda x: float(x.get("score", 0.0)), reverse=True)
        for child in scored_children[:k]:
            dfs(path + [child], depth + 1)
        # Also allow stopping early if the current path is already good.
        consider_final(path, score)

    roots = []
    for thought in initial_thoughts:
        roots.append(thought if isinstance(thought, dict) else {"thought": str(thought)})
    # Prioritize better initial thoughts.
    for root in roots:
        root_score = score_thought(client, question, root, text=text, table=table, images=images, text_model=text_model, vision_model=vision_model)
        root["score"] = float(root_score.get("score", 0.0))
        root["score_rationale"] = root_score.get("rationale", "")
    roots.sort(key=lambda x: float(x.get("score", 0.0)), reverse=True)
    for root in roots[:k]:
        dfs([root], 1)
    return best


In [ ]:
%%writefile Eval.py
"""Evaluation utilities for MAMMQA/Qwen predictions."""

from __future__ import annotations

import argparse
import json
import re
import string
from collections import Counter
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence


def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    records = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(records: Iterable[Dict[str, Any]], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def extract_answer_tag(text: str) -> str:
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", str(text or ""), flags=re.IGNORECASE | re.DOTALL)
    return m.group(1).strip() if m else str(text or "").strip()


def _strip_articles(text: str) -> str:
    return re.sub(r"\b(a|an|the)\b", " ", text)


def normalize_answer(text: Any) -> str:
    text = str(text or "").lower()
    text = _strip_articles(text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def token_f1(prediction: Any, gold: Any) -> float:
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def exact_match(prediction: Any, gold: Any) -> bool:
    return normalize_answer(prediction) == normalize_answer(gold)


_NLP = None

def _lemma_tokens(text: Any) -> List[str]:
    global _NLP
    if _NLP is None:
        try:
            import spacy  # type: ignore
            try:
                _NLP = spacy.load("en_core_web_sm")
            except Exception:
                _NLP = spacy.blank("en")
        except Exception:
            _NLP = False
    norm = normalize_answer(text)
    if not norm:
        return []
    if _NLP is False:
        return norm.split()
    doc = _NLP(norm)  # type: ignore[operator]
    toks = []
    for tok in doc:
        lemma = getattr(tok, "lemma_", "") or tok.text
        lemma = normalize_answer(lemma)
        if lemma:
            toks.append(lemma)
    return toks or norm.split()


def lemma_match(prediction: Any, gold: Any) -> bool:
    p = _lemma_tokens(prediction)
    g = _lemma_tokens(gold)
    return bool(p and g and " ".join(p) == " ".join(g))


def _gold_list(gold: Any) -> List[str]:
    if gold is None:
        return [""]
    if isinstance(gold, list):
        return [str(x) for x in gold]
    if isinstance(gold, str) and " | " in gold:
        return [x.strip() for x in gold.split(" | ") if x.strip()]
    return [str(gold)]


def evaluate_record(record: Dict[str, Any]) -> Dict[str, Any]:
    pred = record.get("predicted_answer") or extract_answer_tag(record.get("Final Answer", ""))
    golds = _gold_list(record.get("gold_answer") or record.get("answer") or record.get("true_answer"))
    em = max(exact_match(pred, g) for g in golds)
    f1 = max(token_f1(pred, g) for g in golds)
    lem = max(lemma_match(pred, g) for g in golds)
    out = dict(record)
    out.update({"predicted_answer": pred, "exact_match": bool(em), "f1": float(f1), "lemma_match": bool(lem)})
    return out


def evaluate_predictions(records: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    evaluated = [evaluate_record(r) for r in records]
    n = len(evaluated) or 1
    return {
        "n": len(evaluated),
        "exact_match": sum(1 for r in evaluated if r["exact_match"]) / n,
        "token_f1": sum(float(r["f1"]) for r in evaluated) / n,
        "lemma_match_accuracy": sum(1 for r in evaluated if r["lemma_match"]) / n,
        "records": evaluated,
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--pred", required=True, help="Prediction JSONL containing predicted_answer and gold_answer fields.")
    parser.add_argument("--out", default="eval_report.json")
    args = parser.parse_args()
    report = evaluate_predictions(read_jsonl(args.pred))
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_agents_qwen.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

from Dataloader import MultiModalQADataLoader
from Eval import evaluate_predictions, write_jsonl
from agents_qwen import get_answer_MM
from qwen_client import DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL, make_qwen_client


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="/kaggle/input/datasets/adibraian/multimodalqa-dataset")
    p.add_argument("--images-base-url", default=None)
    p.add_argument("--encode-images", action="store_true")
    p.add_argument("--n", type=int, default=3)
    p.add_argument("--start", type=int, default=0)
    p.add_argument("--out", default="outputs/qwen_mammqa_predictions.jsonl")
    p.add_argument("--text-model", default=DEFAULT_TEXT_MODEL)
    p.add_argument("--vision-model", default=DEFAULT_VISION_MODEL)
    p.add_argument("--region", default=None)
    p.add_argument("--base-url", default=None)
    p.add_argument("--reduce-modalities-by-type", action="store_true")
    p.add_argument("--not-strict", action="store_true")
    args = p.parse_args()

    client = make_qwen_client(base_url=args.base_url, region=args.region)
    dl = MultiModalQADataLoader.from_root(args.root, images_base_url=args.images_base_url, encode_images=args.encode_images)
    records = []
    end = min(len(dl), args.start + args.n)
    for idx in range(args.start, end):
        ex = dl.get_agent_inputs(idx)
        print(f"\n[{idx}] {ex['type']} :: {ex['question']}")
        res = get_answer_MM(
            client,
            question=ex["question"],
            text=ex["text"],
            tables=ex["table"],
            images=ex["images"],
            text_model=args.text_model,
            vision_model=args.vision_model,
            qtype=ex["type"],
            strict_paper_mode=not args.not_strict,
            reduce_modalities_by_question_type=args.reduce_modalities_by_type,
            verbose=True,
        )
        records.append({
            "id": ex["id"],
            "question": ex["question"],
            "type": ex["type"],
            "method": "qwen_mammqa",
            "gold_answer": ex["answer"],
            **res,
        })
        print("Predicted:", res["predicted_answer"], "| Gold:", ex["answer"])
    write_jsonl(records, args.out)
    report = evaluate_predictions(records)
    report_path = str(Path(args.out).with_suffix(".eval.json"))
    Path(report_path).write_text(json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2), encoding="utf-8")
    print("\nSaved:", args.out)
    print("Report:", json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_baselines_qwen.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

from Dataloader import MultiModalQADataLoader
from Eval import evaluate_predictions, write_jsonl
from agents_qwen import extract_answer_tag, get_answer_cot, get_answer_zs_no_data
from qwen_client import DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL, make_qwen_client


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="/kaggle/input/datasets/adibraian/multimodalqa-dataset")
    p.add_argument("--images-base-url", default=None)
    p.add_argument("--encode-images", action="store_true")
    p.add_argument("--n", type=int, default=3)
    p.add_argument("--start", type=int, default=0)
    p.add_argument("--method", choices=["zs", "cot"], default="cot")
    p.add_argument("--out", default="outputs/qwen_baseline_predictions.jsonl")
    p.add_argument("--text-model", default=DEFAULT_TEXT_MODEL)
    p.add_argument("--vision-model", default=DEFAULT_VISION_MODEL)
    p.add_argument("--region", default=None)
    p.add_argument("--base-url", default=None)
    args = p.parse_args()

    client = make_qwen_client(base_url=args.base_url, region=args.region)
    dl = MultiModalQADataLoader.from_root(args.root, images_base_url=args.images_base_url, encode_images=args.encode_images)
    records = []
    end = min(len(dl), args.start + args.n)
    for idx in range(args.start, end):
        ex = dl.get_agent_inputs(idx)
        if args.method == "zs":
            raw = get_answer_zs_no_data(client, ex["question"], model=args.text_model)
        else:
            raw = get_answer_cot(client, ex["question"], ex["text"], ex["table"], ex["images"], text_model=args.text_model, vision_model=args.vision_model)
        pred = extract_answer_tag(raw)
        print(f"[{idx}] pred={pred!r} gold={ex['answer']!r}")
        records.append({
            "id": ex["id"], "question": ex["question"], "type": ex["type"],
            "method": "qwen_" + args.method, "predicted_answer": pred,
            "gold_answer": ex["answer"], "Final Answer": raw,
        })
    write_jsonl(records, args.out)
    report = evaluate_predictions(records)
    report_path = str(Path(args.out).with_suffix(".eval.json"))
    Path(report_path).write_text(json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2), encoding="utf-8")
    print("Saved:", args.out)
    print(json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_tot_qwen.py
from __future__ import annotations

import argparse
import json

from Dataloader import MultiModalQADataLoader
from Eval import evaluate_predictions, write_jsonl
from qwen_client import DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL, make_qwen_client
from tot_dfs_qwen import run_dfs


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="/kaggle/input/datasets/adibraian/multimodalqa-dataset")
    p.add_argument("--idx", type=int, default=0)
    p.add_argument("--images-base-url", default=None)
    p.add_argument("--encode-images", action="store_true")
    p.add_argument("--k", type=int, default=3)
    p.add_argument("--max-depth", type=int, default=3)
    p.add_argument("--beam-threshold", type=float, default=0.45)
    p.add_argument("--out", default="outputs/qwen_tot_prediction.jsonl")
    p.add_argument("--text-model", default=DEFAULT_TEXT_MODEL)
    p.add_argument("--vision-model", default=DEFAULT_VISION_MODEL)
    p.add_argument("--region", default=None)
    p.add_argument("--base-url", default=None)
    args = p.parse_args()

    client = make_qwen_client(base_url=args.base_url, region=args.region)
    dl = MultiModalQADataLoader.from_root(args.root, images_base_url=args.images_base_url, encode_images=args.encode_images)
    ex = dl.get_agent_inputs(args.idx)
    result = run_dfs(
        client,
        question=ex["question"],
        text=ex["text"],
        table=ex["table"],
        images=ex["images"],
        k=args.k,
        max_depth=args.max_depth,
        beam_threshold=args.beam_threshold,
        text_model=args.text_model,
        vision_model=args.vision_model,
        verbose=True,
    )
    record = {
        "id": ex["id"], "question": ex["question"], "type": ex["type"],
        "method": "qwen_tot", "predicted_answer": result["predicted_answer"],
        "gold_answer": ex["answer"], "Final Answer": result["final_raw"], "tot": result,
    }
    write_jsonl([record], args.out)
    print(json.dumps(evaluate_predictions([record]), ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 3. Configure Qwen / DashScope

Use a Kaggle Secret or environment variable named `DASHSCOPE_API_KEY`.

Region notes:

- `intl` uses `https://dashscope-intl.aliyuncs.com/compatible-mode/v1`
- `us` uses `https://dashscope-us.aliyuncs.com/compatible-mode/v1`
- `beijing` uses `https://dashscope.aliyuncs.com/compatible-mode/v1`

Your DashScope API key must match the region/account where it was created.

In [ ]:
import os
from qwen_client import make_qwen_client, DEFAULT_TEXT_MODEL, DEFAULT_VISION_MODEL

# Change these if needed.
QWEN_REGION = os.getenv("QWEN_REGION", "intl")        # "intl", "us", or "beijing"
QWEN_TEXT_MODEL = os.getenv("QWEN_TEXT_MODEL", "qwen-plus")
QWEN_VISION_MODEL = os.getenv("QWEN_VISION_MODEL", "qwen3-vl-plus")

# Optional: If you prefer to set the secret manually for a quick test, uncomment this line.
# os.environ["DASHSCOPE_API_KEY"] = "your-dashscope-key-here"

client = make_qwen_client(region=QWEN_REGION)

print("Qwen client ready")
print("Region:", QWEN_REGION)
print("Text model:", QWEN_TEXT_MODEL)
print("Vision model:", QWEN_VISION_MODEL)

## 4. Locate and load the MultimodalQA dataset

In [ ]:
from pathlib import Path
from Dataloader import MultiModalQADataLoader, discover_mmqa_files

# Kaggle path from your previous notebook. Add or change paths if your dataset is mounted differently.
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset"),
    Path("/kaggle/input/multimodalqa-dataset"),
    Path("/kaggle/input"),
    Path("data"),
]

DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate.exists():
        files = discover_mmqa_files(candidate)
        if files.get("dev") and files.get("texts") and files.get("tables"):
            DATA_ROOT = candidate
            break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find MMQA files. Set DATA_ROOT to the folder containing MMQA_dev.jsonl, "
        "MMQA_texts.jsonl, MMQA_tables.jsonl, and MMQA_images.jsonl."
    )

print("Using DATA_ROOT:", DATA_ROOT)
print(discover_mmqa_files(DATA_ROOT))

# For local image files, encode_images=True sends data URLs to Qwen-VL.
# If images are already public URLs, False is fine.
dl = MultiModalQADataLoader.from_root(DATA_ROOT, images_base_url=None, encode_images=False)
print("Loaded examples:", len(dl))

ex0 = dl.get_agent_inputs(0)
print("Example 0 keys:", ex0.keys())
print("Question:", ex0["question"])
print("Type:", ex0["type"])
print("Modalities:", ex0["modalities"])
print("Gold answer:", ex0["answer"])

## 5. Run the full Qwen MAMMQA pipeline

`STRICT_PAPER_MODE=True` keeps the three-stage structure.  
`REDUCE_MODALITIES_BY_QUESTION_TYPE=False` means the pipeline uses every available modality. Set it to `True` only when you want a cheaper test run.

In [ ]:
import json
from pathlib import Path
from agents_qwen import get_answer_MM
from Eval import evaluate_predictions, write_jsonl

N_EXAMPLES = 3
START_INDEX = 0
STRICT_PAPER_MODE = True
REDUCE_MODALITIES_BY_QUESTION_TYPE = False  # set True for cheaper single-modality testing

all_results = []
for idx in range(START_INDEX, min(len(dl), START_INDEX + N_EXAMPLES)):
    ex = dl.get_agent_inputs(idx)
    print("
" + "#" * 80)
    print(f"EXAMPLE {idx} | Type: {ex['type']}")
    print("Question:", ex["question"])
    print("Modalities:", ex["modalities"])
    print("#" * 80)

    res = get_answer_MM(
        client=client,
        question=ex["question"],
        text=ex["text"],
        tables=ex["table"],
        images=ex["images"],
        text_model=QWEN_TEXT_MODEL,
        vision_model=QWEN_VISION_MODEL,
        qtype=ex["type"],
        strict_paper_mode=STRICT_PAPER_MODE,
        reduce_modalities_by_question_type=REDUCE_MODALITIES_BY_QUESTION_TYPE,
        verbose=True,
    )

    record = {
        "id": ex["id"],
        "question": ex["question"],
        "type": ex["type"],
        "method": "qwen_mammqa",
        "gold_answer": ex["answer"],
        **res,
    }
    all_results.append(record)
    print("Predicted:", record["predicted_answer"])
    print("Gold:", record["gold_answer"])

report = evaluate_predictions(all_results)
print("
Evaluation summary:")
print(json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2))

## 6. Inspect intermediate agent outputs

In [ ]:
STAGE_KEYS = [
    ("Stage 1 - Text specialist", "Text Agent Output"),
    ("Stage 1 - Table specialist", "Table Agent Output"),
    ("Stage 1 - Image specialist", "Image Agent Output"),
    ("Stage 2 - Text-anchored refinement", "Text Cross Agent Output"),
    ("Stage 2 - Table-anchored refinement", "Table Cross Agent Output"),
    ("Stage 2 - Image-anchored refinement", "Image Cross Agent Output"),
    ("Stage 3 - Final answer", "Final Answer"),
]

EXAMPLE_TO_VIEW = 0
r = all_results[EXAMPLE_TO_VIEW]
print("Question:", r["question"])
print("Gold:", r["gold_answer"])
print("Predicted:", r["predicted_answer"])

for label, key in STAGE_KEYS:
    print("
" + "=" * 80)
    print(label)
    print("=" * 80)
    print(r.get(key, ""))

## 7. Save predictions and evaluation report

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pred_path = OUTPUT_DIR / "qwen_mammqa_predictions.jsonl"
eval_path = OUTPUT_DIR / "qwen_mammqa_eval.json"
csv_path = OUTPUT_DIR / "qwen_mammqa_predictions.csv"

write_jsonl(all_results, pred_path)
eval_path.write_text(json.dumps({k: v for k, v in report.items() if k != "records"}, indent=2), encoding="utf-8")
pd.DataFrame(all_results).to_csv(csv_path, index=False)

print("Saved predictions:", pred_path)
print("Saved eval:", eval_path)
print("Saved CSV:", csv_path)

## 8. Run baselines: zero-shot and CoT

In [ ]:
from agents_qwen import extract_answer_tag, get_answer_cot, get_answer_zs_no_data

RUN_BASELINES = True
baseline_results = []

if RUN_BASELINES:
    for method in ["zs", "cot"]:
        print("
Running baseline:", method)
        for idx in range(START_INDEX, min(len(dl), START_INDEX + N_EXAMPLES)):
            ex = dl.get_agent_inputs(idx)
            if method == "zs":
                raw = get_answer_zs_no_data(client, ex["question"], model=QWEN_TEXT_MODEL)
            else:
                raw = get_answer_cot(
                    client,
                    ex["question"],
                    ex["text"],
                    ex["table"],
                    ex["images"],
                    text_model=QWEN_TEXT_MODEL,
                    vision_model=QWEN_VISION_MODEL,
                )
            pred = extract_answer_tag(raw)
            rec = {
                "id": ex["id"],
                "question": ex["question"],
                "type": ex["type"],
                "method": "qwen_" + method,
                "predicted_answer": pred,
                "gold_answer": ex["answer"],
                "Final Answer": raw,
            }
            baseline_results.append(rec)
            print(f"{method} | idx={idx} | pred={pred!r} | gold={ex['answer']!r}")

    baseline_report = evaluate_predictions(baseline_results)
    print("
Baseline report:")
    print(json.dumps({k: v for k, v in baseline_report.items() if k != "records"}, indent=2))
    write_jsonl(baseline_results, OUTPUT_DIR / "qwen_baselines_predictions.jsonl")

## 9. Optional Tree-of-Thoughts run

This is intentionally off by default because ToT makes many Qwen calls. Turn it on after the main pipeline works.

In [ ]:
RUN_TOT = False

if RUN_TOT:
    from tot_dfs_qwen import run_dfs

    TOT_INDEX = 0
    ex = dl.get_agent_inputs(TOT_INDEX)
    tot_result = run_dfs(
        client,
        question=ex["question"],
        text=ex["text"],
        table=ex["table"],
        images=ex["images"],
        k=3,
        beam_threshold=0.45,
        max_depth=3,
        text_model=QWEN_TEXT_MODEL,
        vision_model=QWEN_VISION_MODEL,
        verbose=True,
    )
    print(json.dumps(tot_result, ensure_ascii=False, indent=2))

## 10. Custom question test

In [ ]:
MY_QUESTION = "Which city hosted the first summer Olympics in the modern era?"
MY_TEXT = """
title: Summer Olympics History
text: The modern Olympic Games were revived in 1896 and held in Athens, Greece.
The 1900 Olympics were held in Paris, France.
"""
MY_TABLE = """
| Year | City      | Country | Nations |
|------|-----------|---------|---------|
| 1896 | Athens    | Greece  | 14      |
| 1900 | Paris     | France  | 24      |
| 1904 | St. Louis | USA     | 12      |
"""
MY_IMAGES = []

RUN_CUSTOM_TEST = False

if RUN_CUSTOM_TEST:
    custom = get_answer_MM(
        client,
        question=MY_QUESTION,
        text=MY_TEXT,
        tables=MY_TABLE,
        images=MY_IMAGES,
        text_model=QWEN_TEXT_MODEL,
        vision_model=QWEN_VISION_MODEL,
        strict_paper_mode=True,
        verbose=True,
    )
    print(custom["Final Answer"])
    print("Extracted answer:", custom["predicted_answer"])

## CLI equivalents

After running the module-writing cells, you can also run these commands in a terminal:

```bash
python run_agents_qwen.py --root /kaggle/input/datasets/adibraian/multimodalqa-dataset --n 3
python run_baselines_qwen.py --method cot --n 3
python run_tot_qwen.py --idx 0 --k 3 --max-depth 3
python Eval.py --pred outputs/qwen_mammqa_predictions.jsonl --out outputs/qwen_mammqa_eval.json
```
